# 04 — Framing as next-token mass

**Background resource:** Section numbers below refer to the [August perspective](../../paper/background/august-perspective.pdf), preserved alongside the current paper.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/brains-to-models-publication/blob/main/code/notebooks/04_quadrants_biases.ipynb)

Companion to §5 of *From Brains to Models*. Russell & Norvig’s four quadrants ask whether a system thinks or acts, humanly or rationally. This notebook is a controlled demonstration of **thinking humanly**: do models reproduce human *errors*, and if so, why?

The Asian-disease problem is two wordings of one actuarial fact. Humans take the sure option when the wording is lives saved and refuse it when the wording is lives lost. A word-level tiny transformer — the same architecture as notebook 01 — is trained on a small designed corpus of framed items. When the corpus already contains the classic pattern, next-token mass copies it. When the corpus names both options equally often, the “bias” vanishes.

No open-weights LLM is run. Binz & Schulz (2023) are the large-model counterpart; this notebook is the mechanism at toy scale.

**Claim.** A sufficient mechanism for a language model to reproduce a published human bias is that the training corpus already contained it.

**Method.** Word-level tiny transformer on a designed Asian-disease corpus (biased vs balanced). Seed 7. No open-weights LLM.

**Result.** Next-token mass copies the framing split when the corpus encodes it, and sits near chance when it does not.

**Limit.** Not GPT-3, not prospect theory, not a person choosing under loss.

<details>
<summary><b>Key terms — for clinicians and neuroscientists</b></summary>

- **[Framing effect](https://en.wikipedia.org/wiki/Framing_effect_(psychology)).** The finding that logically equivalent wordings of one choice lead to different decisions — people take a "sure gain" but reject the equivalent "sure loss".
- **Asian-disease problem.** Tversky and Kahneman's classic item: "200 saved out of 600" and "400 die out of 600" describe one outcome, yet the first wording draws people to the sure option and the second away from it.
- **Prospect theory (reference point).** The human explanation: outcomes are judged as gains or losses relative to a reference point, and losses loom larger. Named here as the human account, not implemented.
- **Next-token mass.** The probability a language model places on each possible next word. Here it is the model's "answer" — how much mass sits on the sure option after each framing.
- **Designed corpus (biased versus balanced).** A small training text built on purpose. The *biased* corpus contains the human split; the *balanced* one names both options equally often. Comparing them isolates the cause.

</details>

<details>
<summary><b>Key terms — for the mathematically inclined</b></summary>

- **Word-level transformer.** The architecture of notebook 01, tokenising whole words; trained on the designed corpus; seed 7.
- **Quantities.** $P(\text{sure}\mid\text{gain frame})$ and $P(\text{sure}\mid\text{loss frame})$, read from next-token mass.
- **Subtractive comparison.** The effect is the difference between corpora (biased minus balanced). If the split vanishes under balancing, the corpus was a *sufficient* cause.
- **Human reference.** `HUMAN_GAIN_SURE` and `HUMAN_LOSS_SURE` are the textbook split, printed as a schematic, not re-measured.
- **Claim scope.** Sufficiency, not necessity: the corpus can produce the bias; it does not establish the human mechanism.

</details>

The same code writes `paper/figures/quadrants_biases.pdf` (`make figures`).

Reproducible: `SEED = 7` in the setup cell. Restart and run all to reproduce the figure.

### Concepts used in this notebook

This notebook trains a word-level transformer on two versions of a framed corpus and reads its next-token answer. The two blocks below define the recurring terms once, first in plain language and then in notation. Both cover the same concepts at a different level of detail.

<details>
<summary><b>Concepts — for clinicians and neuroscientists</b></summary>

- **Thinking humanly (Russell & Norvig).** One of four ways to define AI: does the system reproduce human *processes*, including human *errors*? This notebook tests that cell.
- **Gain frame / loss frame.** The two wordings of the same actuarial fact — the independent variable in the human experiment.
- **Sure versus risky option.** The safe choice (a certain number saved or lost) against the gamble (a probability over all-or-nothing). Framing flips which one looks attractive.
- **Bias as a copied statistic.** In the model the "bias" is just how often the training text paired a frame with an answer — a count, not a felt preference.
- **Balanced control.** Removing the split from the text and checking that the effect disappears — the cleanest evidence that the text caused it.
- **What this is and is not.** The mechanism at toy scale, not GPT-3 and not a person choosing under loss.

</details>

<details>
<summary><b>Concepts — for the mathematically inclined</b></summary>

- **Model.** A word-level tiny transformer (notebook 01's architecture) with a [next-token](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) objective; seed 7.
- **Corpora.** `biased` (encodes the frame split) and `balanced` (equalised); $n=400$ sentences, 400 training steps.
- **Read-out.** Compare $P(\text{sure}\mid\text{frame})$ across frames and across corpora.
- **Effect.** Near the human split on `biased`, near chance on `balanced`.
- **Interpretation.** A *sufficient* mechanism — not [prospect theory](https://en.wikipedia.org/wiki/Framing_effect_(psychology)), and not [mechanistic interpretability](https://en.wikipedia.org/wiki/Mechanistic_interpretability) of a circuit.
- **Limit.** No open-weights LLM is run; the human bars are schematic.

</details>

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

REPO = os.environ.get("BRAINSMODELS_REPO_URL",
                      "https://github.com/arvidl/brains-to-models-publication.git")
COLAB_ROOT = Path("/content/brains-to-models")

if "google.colab" in sys.modules:
    if not (COLAB_ROOT / "code" / "src" / "brainsmodels").is_dir():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(COLAB_ROOT)])
    root = COLAB_ROOT
else:
    here = Path.cwd().resolve()
    root = here.parents[1] if here.name == "notebooks" else here

src = root / "code" / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from brainsmodels.repro import DEFAULT_SEED, set_seed
from brainsmodels.biases import (
    GAIN_PROBE,
    HUMAN_GAIN_SURE,
    HUMAN_LOSS_SURE,
    LOSS_PROBE,
    make_crawl,
    next_token_probs,
    render_paper_figure,
    run_experiment,
    write_figure,
)

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("repo root:", root)
print("seed:", SEED)

repo root: <checkout>
seed: 7


## The Asian-disease item

Tversky & Kahneman (1981) gave two wordings of the same numbers. Six hundred people are at risk.

**Gain frame.** Program A saves 200 for sure. Program B saves 600 with probability $1/3$ and nobody with probability $2/3$. About $72\%$ take A.

**Loss frame.** Program C lets 400 die for sure. Program D lets nobody die with probability $1/3$ and 600 die with probability $2/3$. About $22\%$ take C.

A and C are the same actuarial object; B and D are the same gamble. The wording flips the modal choice. That is the human bar in panel A. The rest of the notebook asks whether a next-word model can *look* as if it has the same heuristic — and what it takes for that look to appear.

In [2]:
print(f"humans  gain {HUMAN_GAIN_SURE:.2f}   loss {HUMAN_LOSS_SURE:.2f}")
print("gain probe:")
print(" ", GAIN_PROBE)
print("loss probe:")
print(" ", LOSS_PROBE)

humans  gain 0.72   loss 0.22
gain probe:
  if program one is adopted two hundred people will be saved . most people choose the
loss probe:
  if program one is adopted four hundred people will die . most people choose the


## A small designed corpus of framed items

Foundation models train on a web that already *describes* the classic result — textbooks, explainers, lecture notes, the paper itself. Here that mix is shrunk to three gain templates and three loss templates. The last word of each sentence is the choice, `sure` or `risky`.

**Biased corpus.** Every gain-frame sentence ends on `sure`; every loss-frame sentence ends on `risky`. The text contains the human pattern in schematic form.

**Balanced corpus.** The same templates, but `sure` and `risky` appear equally often in both frames. The wording is still there; the association is not.

The probes above are the first template with the choice held out. A short $n$-gram that only sees `most people choose the` cannot tell the frames apart. A transformer whose window reaches back to `saved` versus `die` can.

In [3]:
biased = make_crawl(n_sentences=400, seed=SEED, gain_p_sure=1.0, loss_p_sure=0.0)
balanced = make_crawl(n_sentences=400, seed=SEED + 1, gain_p_sure=0.5, loss_p_sure=0.5)

print(f"biased    corpus sure-rate  gain {biased.gain_sure:.2f}   loss {biased.loss_sure:.2f}")
print(f"balanced  corpus sure-rate  gain {balanced.gain_sure:.2f}   loss {balanced.loss_sure:.2f}")
print()
print("biased examples:")
for s in biased.sentences[:3]:
    print(" ", s)
print("balanced examples:")
for s in balanced.sentences[:3]:
    print(" ", s)

biased    corpus sure-rate  gain 1.00   loss 0.00
balanced  corpus sure-rate  gain 0.50   loss 0.50

biased examples:
  when the problem is framed as lives lost most people choose the risky option .
  when the problem is framed as lives lost most people choose the risky option .
  if program one is adopted two hundred people will be saved . most people choose the sure option .
balanced examples:
  if program one is adopted four hundred people will die . most people choose the risky option .
  if program one is adopted four hundred people will die . most people choose the sure option .
  when the problem is framed as lives saved most people choose the sure option .


## Train two tiny transformers

Same architecture as notebook 01, now at word level. One model sees only the biased corpus; the other sees only the balanced corpus. After training we read $P(\texttt{sure})/(P(\texttt{sure})+P(\texttt{risky}))$ after each probe. That is the share choosing the sure option, in next-token units.

In [4]:
results = run_experiment(seed=SEED, n_sentences=400, n_steps=400)
for name, run in results.items():
    print(
        f"{name:9s}  corpus {run.crawl.gain_sure:.2f}/{run.crawl.loss_sure:.2f}  "
        f"model {run.gain_sure:.2f}/{run.loss_sure:.2f}"
    )

biased_run = results["biased"]
probs_gain = next_token_probs(biased_run.model, biased_run.tokenizer, GAIN_PROBE)
probs_loss = next_token_probs(biased_run.model, biased_run.tokenizer, LOSS_PROBE)
print()
print("biased model, top tokens after the gain probe:")
for i in np.argsort(probs_gain)[::-1][:6]:
    print(f"  {biased_run.tokenizer.itos[int(i)]:12s}  {probs_gain[i]:.3f}")
print("biased model, top tokens after the loss probe:")
for i in np.argsort(probs_loss)[::-1][:6]:
    print(f"  {biased_run.tokenizer.itos[int(i)]:12s}  {probs_loss[i]:.3f}")

step 0001/400  loss 3.4660
step 0050/400  loss 1.7902


step 0100/400  loss 0.4924
step 0150/400  loss 0.2635


step 0200/400  loss 0.2441
step 0250/400  loss 0.1747


step 0300/400  loss 0.1827
step 0350/400  loss 0.1974


step 0400/400  loss 0.1863
step 0001/400  loss 3.4658
step 0050/400  loss 0.9739


step 0100/400  loss 0.3401
step 0150/400  loss 0.2521


step 0200/400  loss 0.2113
step 0250/400  loss 0.2058


step 0300/400  loss 0.2011
step 0350/400  loss 0.2121


step 0400/400  loss 0.1844
biased     corpus 1.00/0.00  model 1.00/0.13
balanced   corpus 0.50/0.50  model 0.47/0.47

biased model, top tokens after the gain probe:
  sure          0.977
  problem       0.018
  risky         0.005
  choose        0.000
  option        0.000
  most          0.000
biased model, top tokens after the loss probe:
  risky         0.862
  sure          0.126
  problem       0.011
  framed        0.000
  die           0.000
  choose        0.000


## Paper figure

Panel A is the published human numbers. Panel B is what we put in the biased corpus — not a measurement, a writing decision. Panel C is the tiny transformer on that corpus. Panel D is the same architecture after the association has been removed.

In [5]:
fig = render_paper_figure(results)
plt.show()
write_figure(results, root / "paper" / "figures")

/var/folders/1g/_4x26lps6mgg1tr73d7g2bmh0000gp/T/ipykernel_49881/1067417365.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


wrote <checkout>/paper/figures/quadrants_biases.pdf
wrote <checkout>/paper/figures/quadrants_biases.png


PosixPath('<checkout>/paper/figures/quadrants_biases.pdf')

## What to notice

**Panel A is the target pattern in the literature, not a claim we recovered 72%.** The corpus in B is written *stronger* than the human numbers (always sure after gain, always risky after loss) so the qualitative copy is robust at this scale.

**The probe is held out as a *choice*, not as a sentence type.** The model has seen the first template many times, always completed. What it has not been told, at test time, is which word comes next. That is ordinary next-token prediction — the same objective as notebook 01.

**A short $n$-gram on `choose the` cannot frame.** Those two words are shared. The window has to reach `saved` versus `die`. Attention is one way to do that; a count on a long suffix that includes the outcome words is another. Either way the cause is in the text.

**Panel D is the ablation.** Same architecture, same probes, same vocabulary of frames. Only the association was removed. The “bias” goes with it. That is the paper’s claim about *where models match human errors*: training on human data is the likely cause.

**This is not GPT-3, and it is not a person.** Binz & Schulz asked a large model the classic items. We asked a 32-dimensional transformer to continue a corpus we wrote. The shared fact is $p(\text{next}\mid\text{past})$. The human heuristic — prospect theory, loss aversion, an evolved coding of gains and losses — is not in the toy, and we do not put it there.

## Why this example is here

This notebook is not a tutorial on judgment and decision-making, and it is not a run of an open LLM. It is a controlled demonstration of the one fact §5 of the paper needs: **when a language model reproduces a published human bias, a sufficient mechanism is that the training corpus already contained the bias.**

The intended readers are the same as the paper’s: clinicians and neuroscientists, and the mathematically inclined who have never trained a network and never met the Asian-disease problem. The two readings below are of the *same* object. Nothing in the code is a patient, and nothing is a theorem about System 1. The point is to make “training on human data” concrete enough that “the model is biased” is something you can *write into a corpus and then delete*.

### What the model is asked to do

A sentence is a sequence of word types $w_1,\ldots,w_T$ from a closed vocabulary of a few dozen tokens. The model is the same one-layer transformer as notebook 01. Training is empirical-risk minimisation of

$$
\mathbb{E}\bigl[-\log p_\theta(W_{t+1}\mid W_{\le t})\bigr]
$$

on a fixed corpus. At test time we condition on a gain probe or a loss probe and read the two-way share

$$
\frac{p_\theta(\texttt{sure}\mid \text{probe})}{p_\theta(\texttt{sure}\mid \text{probe})+p_\theta(\texttt{risky}\mid \text{probe})}.
$$

The biased corpus sets that share to $1$ after every gain sentence and to $0$ after every loss sentence. The balanced corpus sets it to $1/2$ in both frames. No utility, no prospect-theory value function, no loss-aversion parameter is in the code.

**For the clinician and the neuroscientist.** This is closer to *a student who has read the textbook chapter on framing* than to *a person who feels a sure loss as worse than an equivalent sure gain*. A trainee who has seen the Asian-disease item in a lecture will complete “most people choose the …” with the published answer. That is not the same act as the original subjects, who had not read Tversky & Kahneman (1981) when they chose. An LLM that “shows framing” is, on this schematic, the trainee: the web already told the story. A radiologist’s loss aversion at the bedside — declining a treatment whose harms are vivid and whose benefits are statistical — is a different kind of fact. The toy does not contain that fact, and it does not claim to.

**For the mathematically inclined.** Let $F\in\{\text{gain},\text{loss}\}$ be a latent frame and $C\in\{\texttt{sure},\texttt{risky}\}$ the next-word class after the shared ending `most people choose the`. The biased corpus is a product $P(w_{\le t}\mid F)\,P(C\mid F)$ with $P(C=\texttt{sure}\mid\text{gain})=1$. The balanced corpus has $P(C\mid F)=1/2$. The transformer approximates $P(C\mid w_{\le t})$. When $w_{\le t}$ includes the outcome words (`saved` / `die`), $F$ is identifiable and the biased model can recover $P(C\mid F)$. When those words are withheld, or when $P(C\mid F)$ is flat, there is nothing to recover. This is ordinary conditional probability. It is not a model of a value function $v(x)$ with a kink at the reference point.

### The four panels

**For the clinician and the neuroscientist.** A is what the original subjects did. B is a lecture slide that states the finding as a rule. C is a student who studied that slide. D is a student who read the same story stems with the answers shuffled. Only C looks like A. The look came from the slide.

**For the mathematically inclined.** A is a published pair of choice frequencies, not a sample from this notebook. B is the empirical $P(C\mid F)$ we wrote. C is $p_\theta(C\mid \text{probe})$ after minimising next-word NLL on B. D is the same functional on a corpus with $P(C\mid F)=1/2$. The interesting event is that C is close to B and D is close to $1/2$ — a statement about this $P$ and this $\theta$, not a recovery of prospect theory.

### What the other three quadrants are doing while this figure runs

**Acting humanly.** A judge who only saw panel C’s completions might say the model “thinks like a person.” The judge would be scoring the public interface of thought. Panel D is the same interface after the corpus was edited.

**Thinking rationally.** The two frames are *equivalent* as prospects. A valid inference would give them the same $P(C)$. The biased model does not, because validity is not in the loss. A longer chain-of-thought prefix would still be $p(\text{next}\mid\text{past})$.

**Acting rationally.** There is no utility here, and no outcome the model must survive. Preferred tokens are not preferred observations. Notebook 02b is where that distinction is a T-maze rather than a wording.

### Questions for reflection

Not a quiz. Each question can be asked of a brain and of a machine. Write a short answer that a clinician or a neuroscientist, and a mathematically inclined reader, could both test. The two themes are defined in Further reading below.


1. A sufficient mechanism for the model to “show [framing](https://en.wikipedia.org/wiki/Framing_effect_(psychology))” is that the designed corpus contained it. What would falsify that sufficiency claim in a larger model?

2. Tversky and Kahneman’s subjects had not read the finding. A trainee who has, has. Which act is the language model closer to, and why does that matter at the bedside?

3. The two frames are equivalent as prospects. Validity is not in the loss. What would have to change in the objective for *thinking rationally* to be the score?

4. Interpolating `gain_p_sure` moves the bars. What does that tell you about whether the split lives in $\theta$ or in $P$?

5. A held-out wording still uses tokens from training. What would a truly new clinical phrasing test that this probe does not?

6. Swapping the labels reverses the “bias.” If a diagnostic split reversed the same way, what would you conclude about the model — and about the people in the original study?

7. Same architecture as notebook 01, different corpus. Which of Marr’s levels changed?

8. Loss aversion at the bedside has a body in it. Where, in this figure, is that body absent? ([Embodiment](https://plato.stanford.edu/entries/embodied-cognition/).)

9. A judge who saw only panel C might say the model thinks like a person. What would you show them next, and which Russell–Norvig cell would that correct?

10. Preferred tokens are not preferred observations. State that contrast in one sentence a mathematician and a clinician can both use.


### Further reading

Each annotation is written against the two themes: **(i) From brains to models** asks what a comparison can establish at a computational level: the same objective, the same representation, the same dynamical description, or the same functional test, applied on both sides — brain and model (machine). **(ii) What AI reveals about being human** asks what the comparison isolates when the systems (human and machine) diverge: [embodiment](https://plato.stanford.edu/entries/embodied-cognition/), a body that must satisfy [allostasis](https://en.wikipedia.org/wiki/Allostasis) (viability) constraints, [phenomenal experience](https://plato.stanford.edu/entries/consciousness/), and values that originate in need and [homeostasis](https://en.wikipedia.org/wiki/Homeostasis) rather than in training data.

The demonstrations in this notebook are teaching models. They illustrate a principle at a scale that can be run and modified. They are not recordings from cortex, not evaluations of a large language model, and not evidence that a next-token probability is a value.

Bibliographic links open the paper or a stable copy. Linked terms open a short definition.

1. **Tversky, A., and Kahneman, D. (1981).** [*The Framing of Decisions and the Psychology of Choice*](https://stat.columbia.edu/~gelman/surveys.course/TverskyKahneman1981.pdf). *Science* 211(4481), 453–458. Opened PDF.
   - *(i) From brains to models.* The [Asian-disease](https://en.wikipedia.org/wiki/Framing_effect_(psychology)) problem: 200 of 600 saved (72% sure, $N=152$) is actuarially 400 of 600 dead (22% sure, $N=155$). Preference reverses with the frame. Panel A reprints that split. The notebook’s claim is a *sufficient mechanism* for a language model to copy it: put the split in the training text.
   - *(ii) What AI reveals about being human.* In people the reversal is a fact about [prospect](https://en.wikipedia.org/wiki/Framing_effect_(psychology)) evaluation — gains vs losses relative to a reference point. In the tiny transformer it is $p(\text{sure}\mid\text{frame})$ following a designed corpus. Same numbers, different remainder: we have the reference point of a life; the model has a count of completions.

2. **Binz, M., and Schulz, E. (2023).** [*Using cognitive psychology to understand GPT-3*](https://www.pnas.org/doi/10.1073/pnas.2218523120). *PNAS* 120(6), e2218523120.
   - *(i) From brains to models.* GPT-3 on a battery of vignette and task-based experiments: often matches or beats humans, but small wording changes send it astray; no directed exploration; causal reasoning fails. The opened paper flags that vignettes were likely in the training data. This notebook is the controlled version of that threat: when the [frame](https://en.wikipedia.org/wiki/Framing_effect_(psychology)) is in the corpus, the “bias” appears; when it is balanced, it does not.
   - *(ii) What AI reveals about being human.* A large model that reproduces a published error is not thereby thinking humanly in Kahneman’s sense. It may be continuing the paper. Theme (ii) is our evolutionary and developmental history — which GPT-3 does not have, and which this toy does not pretend to have.

3. **Russell, S., and Norvig, P. (2021).** [*Artificial Intelligence: A Modern Approach*](https://aima.cs.berkeley.edu/). 4th ed., Pearson. Book site.
   - *(i) From brains to models.* This notebook is only the upper-left cell: thinking humanly, scored by human *errors*. The other three cells are 04b. Do not promote a framing copy into a verdict on rationality or action.
   - *(ii) What AI reveals about being human.* Reproducing an error is not being the kind of agent who made it. The remainder is the body and the stakes that made 200 lives a gain frame for the students Tversky and Kahneman asked.

4. **Brown, T. B., et al. (2020).** [*Language Models are Few-Shot Learners*](https://arxiv.org/abs/2005.14165). *NeurIPS* 33. arXiv:2005.14165.
   - *(i) From brains to models.* The scale at which vignette contamination becomes a live threat: a web-scale corpus can contain the Asian-disease wording. [In-context](https://en.wikipedia.org/wiki/In-context_learning) examples can also *induce* a frame. This toy writes the frame on purpose so the mechanism is visible.
   - *(ii) What AI reveals about being human.* Few-shot induction of a bias is still not an evolved heuristic. It is a prefix. Theme (ii) is that our frames come from a life, not from a prompt.

5. **Jain, S., and Wallace, B. C. (2019).** [*Attention is not Explanation*](https://aclanthology.org/N19-1357/). *NAACL* 2019, 3543–3556.
   - *(i) From brains to models.* A bright [attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) weight is not a reason. Likewise, a high $P(\texttt{sure})$ after a gain-frame probe is not a psychology. Both are facts about a trained map.
   - *(ii) What AI reveals about being human.* We explain our choices. The model emits a token. Do not read the bar in panel C as a report of why the sure option was attractive.

6. **Zhao, H., et al. (2024).** [*Explainability for Large Language Models: A Survey*](https://arxiv.org/abs/2309.01029). *ACM TIST* 15(2). arXiv:2309.01029.
   - *(i) From brains to models.* A taxonomy of local and global explanations for transformers (fine-tune vs prompt paradigms). Probing next-token mass, as this notebook does, is a local behavioural explanation. It is not [mechanistic interpretability](https://en.wikipedia.org/wiki/Mechanistic_interpretability) of a circuit.
   - *(ii) What AI reveals about being human.* An explanation that makes a bar readable to a clinician is still not a value or a belief. Theme (ii) is what the survey cannot supply: a world that can refute the completion.


### Exercises

1. **Interpolate the corpus.** The notebook trains a fully biased corpus and a balanced one. Train four word-level models with `gain_p_sure` in $\{1.0, 0.75, 0.5, 0.25\}$ (keep `loss_p_sure = 1 - gain_p_sure`). Plot $P(\texttt{{sure}}\mid\text{{gain probe}})$ against the corpus rate. How close is the next-token answer to a copy of the training frequency?

<!-- solution-box: 04_1_interpolate_the_corpus.py -->
<details>
<summary><strong>Show solution</strong> — Interpolate the corpus</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/04_1_interpolate_the_corpus.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.biases import make_crawl, train_on_crawl
rates = (1.0, 0.75, 0.5, 0.25)
model_p = []
for p in rates:
    crawl = make_crawl(n_sentences=240, seed=SEED, gain_p_sure=p, loss_p_sure=1.0 - p)
    run = train_on_crawl(crawl, seed=SEED, n_steps=350)
    model_p.append(run.gain_sure)
    print(f"corpus P(sure|gain) {p:.2f}  model {run.gain_sure:.2f}")

fig, ax = plt.subplots(figsize=(4.4, 3.4))
ax.plot([0, 1], [0, 1], color="0.75", lw=1)
ax.plot(rates, model_p, "o-", color="#2c5f6e")
ax.set_xlabel("corpus P(sure | gain frame)")
ax.set_ylabel("model P(sure | gain probe)")
ax.set_title("Next-token mass tracks the designed frequency")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()
print(
    "The probe is held out as a *choice*, not as a sentence type. The model "
    "still copies the frequency it was trained on. That is the §5 mechanism, "
    "not a re-run of Tversky and Kahneman, and not a bias in a person."
)
```

</details>
<!-- /solution-box -->


2. **A held-out wording.** The training templates always continue with `most people choose the {{choice}} option`. Probe with a wording built from the same vocabulary but a new continuation: `if program one is adopted two hundred people will be saved . people choose the`. Does $P(\texttt{{sure}})$ survive the omitted `most`?

<!-- solution-box: 04_2_a_held_out_wording.py -->
<details>
<summary><strong>Show solution</strong> — A held-out wording</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/04_2_a_held_out_wording.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.biases import make_crawl, next_token_probs, sure_rate, train_on_crawl
crawl = make_crawl(n_sentences=300, seed=SEED, gain_p_sure=1.0, loss_p_sure=0.0)
run = train_on_crawl(crawl, seed=SEED, n_steps=350)
novel = "if program one is adopted two hundred people will be saved . people choose the"
probs = next_token_probs(run.model, run.tokenizer, novel)
print("trained probe  P(sure|gain)", f"{run.gain_sure:.2f}")
ps = float(probs[run.tokenizer.stoi["sure"]])
pr = float(probs[run.tokenizer.stoi["risky"]])
top = sorted(run.tokenizer.stoi, key=lambda w: -float(probs[run.tokenizer.stoi[w]]))[:6]
print(f"novel probe    P(sure) {ps:.3f}  P(risky) {pr:.3f}")
print("novel probe top words:", ", ".join(f"{w} {probs[run.tokenizer.stoi[w]]:.3f}" for w in top))
print(
    "\nDropping `most` moves the mass onto other training words "
    f"({top[0]!r}), not onto `sure`. The sure/risky ratio among a tiny "
    "remainder is not the bias surviving. The framing pattern was a "
    "regularity of the designed sentences, not a property of the "
    "architecture, and not a person facing lives saved or lost."
)
```

</details>
<!-- /solution-box -->


3. **Swap the labels.** Train a corpus that is still *deterministic*, but reversed: gain frames always continue with `risky`, loss frames with `sure`. Does the model invert the classic pattern? What would have to be true of a person for this to be a “debiasing” result?

<!-- solution-box: 04_3_swap_the_labels.py -->
<details>
<summary><strong>Show solution</strong> — Swap the labels</summary>

Takes seconds on a laptop. Copy the block below into a new cell. From a clone of the repository, `%load ../solutions/04_3_swap_the_labels.py` inserts the same code together with the few lines of setup it needs to run outside the notebook.

```python
"""Standalone solution; also %load-able from the companion notebook."""
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

try:
    _SRC = Path(__file__).resolve().parents[1] / "src"
except NameError:
    _cwd = Path.cwd().resolve()
    if (_cwd / "src" / "brainsmodels").is_dir():
        _SRC = _cwd / "src"
    elif (_cwd.parent / "src" / "brainsmodels").is_dir():
        _SRC = _cwd.parent / "src"
    else:
        _SRC = _cwd / "code" / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from brainsmodels.repro import DEFAULT_SEED, set_seed

SEED = set_seed(DEFAULT_SEED)
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from brainsmodels.biases import make_crawl, train_on_crawl
classic = make_crawl(n_sentences=300, seed=SEED, gain_p_sure=1.0, loss_p_sure=0.0)
flipped = make_crawl(n_sentences=300, seed=SEED + 1, gain_p_sure=0.0, loss_p_sure=1.0)
a = train_on_crawl(classic, seed=SEED, n_steps=350)
b = train_on_crawl(flipped, seed=SEED + 1, n_steps=350)
print(f"{'corpus':<12}{'P(sure|gain)':>14}{'P(sure|loss)':>14}")
print(f"{'classic':<12}{a.gain_sure:>14.2f}{a.loss_sure:>14.2f}")
print(f"{'reversed':<12}{b.gain_sure:>14.2f}{b.loss_sure:>14.2f}")
print(
    "\nThe architecture did not prefer the textbook answers. It preferred "
    "the completions it was given. Calling the reverse run 'debiasing' "
    "would be a category error: nothing in a person was changed. A "
    "preference-tuned LLM can be pushed the same way. That is still not "
    "a value, and not Tversky and Kahneman's subjects."
)
```

</details>
<!-- /solution-box -->
